# UndertriAI — GRPO Training (Kaggle/Colab)

3-level difficulty curriculum for bail assessment using Qwen2.5-7B-Instruct + Unsloth GRPO.

| Level | Cases | Episodes | Steps | LR | Beta |
|-------|-------|----------|-------|----|------|
| **Easy** | Landmark clear-cut | 104 | 70 | 2e-5 | 0.03 |
| **Medium** | Contested judgment calls | 761 | 180 | 3e-5 | 0.04 |
| **Hard** | Bias reversal + schema drift | 335 | 90 | 2e-5 | 0.02 |

**Training mode:** Online — rewards computed via the live HF Space environment API.
**API reliability:** Fields are canonicalized before sending to prevent 422 errors.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl datasets wandb matplotlib

In [ ]:
# Cell 2: Clone repo and verify data
import os

REPO = "https://huggingface.co/spaces/Draken1606/undertrial-ai"
if not os.path.exists("data/episodes"):
    !git clone {REPO} undertrial_repo
    os.chdir("undertrial_repo")

import glob
for f in sorted(glob.glob("data/episodes/episodes_stage_*.jsonl")):
    lines = sum(1 for _ in open(f))
    print(f"  {f}: {lines} episodes")

In [ ]:
# Cell 3: Configure environment URL
ENV_URL = "https://draken1606-undertrial-ai.hf.space"  # <-- UPDATE THIS

import urllib.request, json
try:
    resp = urllib.request.urlopen(f"{ENV_URL}/health", timeout=10)
    print(f"\u2705 Environment is live: {json.loads(resp.read())}")
except Exception as e:
    print(f"\u26a0\ufe0f Cannot reach {ENV_URL}: {e}")
    print("Falling back to offline mode")
    ENV_URL = None

In [ ]:
# Cell 4: Import training functions
import sys; sys.path.insert(0, '.')
from training.train_grpo import (
    load_episodes, train_curriculum, DIFFICULTY_MAP, DIFFICULTY_NAMES,
    SYSTEM_PROMPT,
)
print("[OK] Imported from train_grpo.py")
for k, v in DIFFICULTY_MAP.items():
    print(f"  {k}: stages={v['stages']}, steps={v['steps']}, lr={v['lr']}, beta={v['beta']}")

In [ ]:
# Cell 5: Verify episode loading + reward fixes
for diff in ['easy', 'medium', 'hard']:
    eps = load_episodes('./data/episodes', difficulty=diff)
    granted = sum(1 for e in eps if 'grant' in e['ground_truth']['outcome'].lower())
    print(f"{diff:8s}: {len(eps)} episodes ({granted} granted, {len(eps)-granted} denied)")

from server.reward import compute_outcome_match, compute_flight_risk_accuracy
import json
ep = json.loads(open('data/episodes/episodes_stage_1.jsonl').readline())
gt = ep['ground_truth']
print(f"\nReward fix check:")
print(f"  Empty flight_risk = {compute_flight_risk_accuracy('', gt):.2f} (should be 0.00)")
print(f"  Wrong direction   = {compute_outcome_match('Bail Denied', gt):.2f} (should be -0.30)")

In [ ]:
# Cell 6: Train! (340 steps: 70 easy + 180 medium + 90 hard)
# Per-level LR/beta schedule from DIFFICULTY_MAP:
#   Easy:   lr=2e-5, beta=0.03 (gentle — 7B already scores ~53% zero-shot)
#   Medium: lr=3e-5, beta=0.04 (bulk learning on contested cases)
#   Hard:   lr=2e-5, beta=0.02 (fine-tuning on bias/schema drift)
results = train_curriculum(
    episodes_dir="./data/episodes",
    output_dir="./output/undertrial_grpo",
    difficulties=["easy", "medium", "hard"],
    model_name="unsloth/Qwen2.5-7B-Instruct",
    wandb_disabled=True,
    env_url=ENV_URL,
)

In [ ]:
# Cell 7: Display results
from pathlib import Path
from IPython.display import Image, display

results_path = Path("./output/undertrial_grpo/curriculum_results.json")
if results_path.exists():
    import json
    data = json.loads(results_path.read_text())
    print("=== CURRICULUM RESULTS ===")
    for level, r in data.get("levels", {}).items():
        print(f"  {level}: {r['baseline']:.4f} \u2192 {r['post']:.4f} (\u0394 = {r['delta']:+.4f})")

for img in ['reward_curve_all_levels.png', 'reward_curve_easy.png',
            'reward_curve_medium.png', 'reward_curve_hard.png',
            'before_after_comparison.png']:
    p = Path(f"./output/undertrial_grpo/plots/{img}")
    if p.exists():
        print(f"\n--- {img} ---")
        display(Image(filename=str(p)))

In [ ]:
# Cell 8: SAVE TO KAGGLE OUTPUT (don't skip!)
import shutil
from pathlib import Path

dst = Path("/kaggle/working/undertrial_results")
dst.mkdir(exist_ok=True)

for png in Path("./output/undertrial_grpo").rglob("*.png"):
    shutil.copy2(str(png), str(dst / png.name))
for j in Path("./output/undertrial_grpo").rglob("*.json"):
    shutil.copy2(str(j), str(dst / j.name))
if Path("./output/undertrial_grpo/final").exists():
    shutil.copytree("./output/undertrial_grpo/final", str(dst / "final_model"), dirs_exist_ok=True)

print(f"\u2705 Saved to {dst}")
for f in sorted(dst.iterdir()):
    print(f"  {f.name}")

In [ ]:
# Cell 9: Merge LoRA (optional)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("./output/undertrial_grpo/final", max_seq_length=2048)
model.save_pretrained_merged("./output/undertrial_grpo/merged", tokenizer, save_method="merged_16bit")
print("Merged model saved")